# Vulnerable Road User Detection: a Safety-Critical Case Study

**Module 01 · Perception Systems** · Notebook 5 of 8

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milinpatel07/Autonomous-Driving_AI-Safety-and-Security/blob/main/01_Perception/05_vulnerable_road_user_detection.ipynb)

| Level | Time | Prerequisites | Standards |
|---|---|---|---|
| Intermediate | ~75 min | [04 Object detection](04_object_detection.ipynb) | ISO 26262-3:2018; ISO 21448:2022; ISO/PAS 8800:2024 |

**Author:** Milin Patel · Hochschule Kempten · MIT Licence

## Learning objectives

By the end of this notebook you will be able to:

1. Explain why detecting vulnerable road users (VRUs) is safety-critical, using sourced accident data
2. Measure how a pretrained detector degrades under darkness, fog and occlusion on a real street image
3. Determine ASILs correctly with the ISO 26262-3 risk graph and keep the ISO 26262 / ISO 21448 / ISO/PAS 8800 boundary sharp
4. Choose an operating threshold from a precision–recall curve for a recall target
5. Translate per-frame recall into distance travelled without seeing a pedestrian, and relate it to stopping distance
6. Distinguish frame-level from track-level recall and propose mitigations

## Why VRU detection is critical

**Vulnerable road users** (pedestrians, cyclists, users of wheelchairs and e-scooters) have no protective shell, can move unpredictably (children, distracted adults), present a small visual and LiDAR signature at distance, and suffer severe injuries even at moderate impact speeds.

### Real-world incidents
- **Uber ATG, Tempe (March 2018):** a pedestrian walking a bicycle was detected about 5.6 s before impact but repeatedly reclassified (vehicle / bicycle / other), so her path was never predicted; the factory AEB was disabled and the safety operator distracted. Fatal collision ([NTSB HAR-19/03](https://www.ntsb.gov/investigations/AccidentReports/Reports/HAR1903.pdf)).
- **Cruise, San Francisco (October 2023):** a driverless Cruise AV struck a pedestrian who had been thrown into its path by a hit-and-run human driver, then dragged her about 20 ft during a pull-over manoeuvre; California permits were suspended and NHTSA imposed a US\$1.5 M penalty for incomplete crash reporting ([NHTSA consent order](https://www.nhtsa.gov/press-releases/consent-order-cruise-crash-reporting)).
- **Tesla FSD (2024–2026):** NHTSA preliminary evaluation PE24-031 (opened October 2024) examines crashes in reduced roadway visibility (sun glare, fog, airborne dust), including one in which a pedestrian was fatally struck; it was upgraded to Engineering Analysis EA26-002 in March 2026 (as of October 2026: open) ([NHTSA PE24-031](https://static.nhtsa.gov/odi/inv/2024/INOA-PE24031-23232.pdf); [Electrek, Mar 2026](https://electrek.co/2026/03/19/nhtsa-upgrades-tesla-fsd-visibility-investigation-3-2-million-vehicles/)).

### Which standard applies?
- **ISO 26262-3:2018:** safety goals for, e.g., AEB or ADS braking towards pedestrians are often rated up to **ASIL D** in a hazard analysis and risk assessment (HARA). ASILs belong to *hazardous events and safety goals*, not to a function such as "pedestrian detection".
- **ISO 21448:2022 (SOTIF)** covers misses caused by **performance limitations** of a correctly working detector (darkness, glare, occlusion).
- **ISO/PAS 8800:2024** addresses insufficiencies specific to the **ML model and its data** (e.g. too few wheelchair users in the training set).

## Setup

Same approach as notebook 04: Ultralytics is installed automatically only on Colab; without it the notebook uses stored detector outputs.

In [ ]:
import sys, importlib.util, subprocess, time
from io import BytesIO
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from IPython.display import display

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and importlib.util.find_spec("ultralytics") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=False)
try:
    from ultralytics import YOLO
    YOLO_AVAILABLE = True
except Exception as e:
    YOLO_AVAILABLE = False
    print(f"Ultralytics not available ({type(e).__name__}); using stored detector outputs.")

rng = np.random.default_rng(42)
OKABE_ITO = ['#E69F00', '#56B4E9', '#009E73', '#F0E442', '#0072B2', '#D55E00', '#CC79A7', '#000000']
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3})
print(f"NumPy {np.__version__} | Ultralytics available: {YOLO_AVAILABLE}")

---
## 1. Pedestrian fatalities and light conditions (sourced data)

NHTSA *Traffic Safety Facts 2022 — Pedestrians* (DOT HS 813 590) reports the light condition for US pedestrian fatalities in 2022.

In [ ]:
# NHTSA Traffic Safety Facts 2022 (DOT HS 813 590): US pedestrian fatalities by light condition
light = pd.DataFrame({'Condition': ['Dark', 'Daylight', 'Dusk', 'Dawn'], 'Percentage': [78, 19, 2, 2]})
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(light['Condition'], light['Percentage'], color=[OKABE_ITO[4], OKABE_ITO[0], OKABE_ITO[6], OKABE_ITO[1]])
for i, p in enumerate(light['Percentage']):
    ax.text(i, p + 1, f"{p} %", ha='center')
ax.set_ylabel('Share of pedestrian fatalities (%)'); ax.set_xlabel('Light condition')
ax.set_title('US pedestrian fatalities by light condition, 2022 (NHTSA DOT HS 813 590)')
ax.set_ylim(0, 90); plt.tight_layout(); plt.show()

dark = int(light.loc[light.Condition == 'Dark', 'Percentage'].iloc[0])
print("--- Light condition (text summary) ---")
for _, r in light.iterrows():
    print(f"  {r['Condition']}: {r['Percentage']} %")
print(f"  ~{dark} % of US pedestrian fatalities in 2022 occurred in the dark (percentages are rounded and sum to {light.Percentage.sum()}).")
print("  Caveat: this is crash data for human-driven traffic; it shows where the risk is, not how a detector performs.")

VRUs are not one homogeneous class. Sub-groups that detectors often find harder include children (small, partly occluded by parked cars), wheelchair users and people pushing objects (unusual silhouettes), and people in dark clothing at night. NHTSA reports pedalcyclists separately from pedestrians.

---
## 2. Detector robustness on a real street image

We take the Ultralytics sample street image `bus.jpg`, synthesise three **degradations** — darkening with sensor noise (a crude night proxy), a fog-like contrast loss, and a vertical occluder — and run YOLO11n on each.

**Reference boxes:** we use the four person boxes found on the clean image as *pseudo ground truth*. The numbers below therefore measure **consistency under degradation**, not absolute accuracy.

> **Illustrative data:** the degradations are synthetic image transforms, not real night or fog photographs.

In [ ]:
def load_image(url, timeout=15):
    try:
        import requests
        from PIL import Image
        r = requests.get(url, timeout=timeout); r.raise_for_status()
        return np.array(Image.open(BytesIO(r.content)).convert('RGB'))
    except Exception as e:
        print(f"Image download failed ({type(e).__name__}); using a placeholder canvas.")
        return None

img = load_image("https://ultralytics.com/images/bus.jpg")
IMAGE_IS_REAL = img is not None
if not IMAGE_IS_REAL:
    img = np.full((1080, 810, 3), 200, np.uint8)

def night(im, gain):
    r = np.random.default_rng(42)
    x = 255 * ((im / 255.0) ** 2.2) * gain + r.normal(0, 4, im.shape)    # gamma, exposure drop, sensor noise
    return np.clip(x, 0, 255).astype(np.uint8)

def fog(im, a):
    return np.clip(im * (1 - a) + a * 200, 0, 255).astype(np.uint8)       # blend towards a grey veil

def occlude(im):
    o = im.copy(); W = im.shape[1]; o[:, W // 3: W // 2] = 80; return o   # vertical occluder (e.g. a pole/van)

CONDITIONS = {
    'normal': lambda im: im,
    'night_0.12': lambda im: night(im, 0.12), 'night_0.06': lambda im: night(im, 0.06), 'night_0.03': lambda im: night(im, 0.03),
    'fog_0.6': lambda im: fog(im, 0.6), 'fog_0.8': lambda im: fog(im, 0.8), 'fog_0.9': lambda im: fog(im, 0.9),
    'occluded': occlude,
}
print(f"Image loaded: {IMAGE_IS_REAL}; conditions: {list(CONDITIONS)}")

In [ ]:
# Stored YOLO11n person detections [confidence, [x1, y1, x2, y2]] (Ultralytics 8.4.191, CPU, conf >= 0.05)
# for exactly the transforms above. Used ONLY if live inference is not possible.
FALLBACK = {
    'normal': [[0.888, [671.0, 394.8, 809.8, 878.7]], [0.878, [47.4, 399.6, 239.3, 904.2]], [0.856, [223.1, 408.7, 344.5, 860.4]], [0.622, [0.0, 556.1, 68.9, 872.4]], [0.092, [0.2, 601.3, 49.9, 870.6]]],
    'night_0.12': [[0.865, [32.3, 399.7, 232.8, 914.4]], [0.808, [224.8, 409.2, 344.5, 862.2]], [0.695, [671.1, 389.7, 809.9, 874.2]], [0.427, [0.0, 551.1, 68.6, 875.4]], [0.106, [0.2, 580.2, 52.1, 876.0]]],
    'night_0.06': [[0.822, [30.6, 393.0, 228.8, 914.7]], [0.655, [217.8, 408.0, 345.2, 860.2]], [0.345, [668.2, 301.4, 809.9, 869.9]], [0.263, [0.1, 551.0, 87.6, 884.5]], [0.179, [0.2, 554.1, 55.3, 877.4]], [0.089, [233.7, 405.1, 342.1, 561.4]], [0.08, [673.7, 524.4, 809.7, 873.5]], [0.062, [225.3, 406.6, 435.2, 860.3]]],
    'night_0.03': [],
    'fog_0.6': [[0.893, [47.0, 399.1, 240.8, 906.3]], [0.875, [223.5, 405.8, 344.1, 860.3]], [0.851, [670.8, 388.3, 809.8, 877.0]], [0.591, [0.0, 552.9, 68.0, 873.0]], [0.091, [787.2, 371.1, 809.6, 466.8]], [0.063, [0.2, 584.0, 46.7, 871.5]]],
    'fog_0.8': [[0.892, [48.6, 398.5, 240.3, 905.9]], [0.856, [226.1, 405.2, 345.6, 859.9]], [0.78, [669.6, 382.3, 809.9, 876.0]], [0.483, [0.1, 550.9, 77.7, 875.5]], [0.277, [0.2, 549.2, 54.7, 873.5]], [0.06, [787.2, 365.3, 809.6, 475.9]]],
    'fog_0.9': [[0.878, [47.2, 397.8, 234.8, 906.9]], [0.844, [225.0, 404.8, 345.2, 859.7]], [0.436, [669.5, 370.8, 809.9, 872.4]], [0.328, [0.1, 549.0, 82.6, 882.8]], [0.158, [0.0, 241.6, 94.0, 889.2]], [0.081, [0.1, 246.0, 42.5, 742.0]], [0.064, [0.1, 247.5, 49.8, 881.3]], [0.055, [0.1, 246.6, 34.2, 325.5]]],
    'occluded': [[0.88, [671.2, 394.8, 809.8, 877.8]], [0.863, [49.1, 396.7, 238.5, 903.2]], [0.611, [223.3, 490.3, 270.5, 853.5]], [0.56, [0.0, 556.3, 68.8, 873.1]], [0.061, [0.2, 605.3, 50.4, 870.7]]]
}

model = None
if YOLO_AVAILABLE and IMAGE_IS_REAL:
    try:
        model = YOLO('yolo11n.pt')
    except Exception as e:
        print(f"Model could not be loaded ({type(e).__name__}); using stored outputs.")

def detect_persons(im, cond):
    """Return list of [conf, box] for class 'person' (conf >= 0.05) and the source used."""
    if model is not None:
        try:
            r = model.predict(np.ascontiguousarray(im[..., ::-1]), conf=0.05, verbose=False)[0]
            return [[float(s), [float(v) for v in b]] for c, s, b in zip(r.boxes.cls, r.boxes.conf, r.boxes.xyxy)
                    if model.names[int(c)] == 'person'], 'live'
        except Exception as e:
            print(f"Inference failed ({type(e).__name__}); using stored output for {cond}.")
    return FALLBACK[cond], 'stored'

results, sources = {}, set()
for cond, fn in CONDITIONS.items():
    results[cond], src = detect_persons(fn(img), cond)
    sources.add(src)
print(f"Detections source: {', '.join(sorted(sources))}")

In [ ]:
def iou(a, b):
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0

def evaluate(dets, refs, conf_thr=0.25, iou_thr=0.5):
    """Greedy one-to-one matching; returns TP, FP, FN and mean confidence of TPs."""
    dets = sorted([d for d in dets if d[0] >= conf_thr], key=lambda d: -d[0])
    used, tp, confs = set(), 0, []
    for c, b in dets:
        cand = [(iou(b, r), j) for j, r in enumerate(refs) if j not in used]
        if cand:
            best, j = max(cand)
            if best >= iou_thr:
                used.add(j); tp += 1; confs.append(c)
    return tp, len(dets) - tp, len(refs) - tp, (np.mean(confs) if confs else np.nan)

REF = [b for c, b in results['normal'] if c >= 0.5]      # pseudo ground truth from the clean image
rows = []
for cond, dets in results.items():
    tp, fp, fn, mc = evaluate(dets, REF)
    rows.append({'condition': cond, 'TP': tp, 'FP': fp, 'FN': fn, 'recall': tp / len(REF),
                 'mean conf (TP)': round(mc, 3) if not np.isnan(mc) else None})
vru_df = pd.DataFrame(rows); display(vru_df)

show = ['normal', 'night_0.06', 'fog_0.9', 'occluded']
fig, axes = plt.subplots(1, 4, figsize=(18, 6.5))
for ax, cond in zip(axes, show):
    ax.imshow(CONDITIONS[cond](img)); ax.axis('off')
    for c, (x1, y1, x2, y2) in results[cond]:
        if c >= 0.25:
            ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, lw=2.5, edgecolor=OKABE_ITO[5]))
            ax.text(x1 + 3, y1 + 20, f"{c:.2f}", color='white', fontsize=9, bbox=dict(facecolor=OKABE_ITO[5], pad=1))
    r = vru_df.set_index('condition').loc[cond]
    ax.set_title(f"{cond}: recall {r['recall']:.2f} (TP {int(r['TP'])}, FN {int(r['FN'])}, FP {int(r['FP'])})")
plt.tight_layout(); plt.show()

print(f"--- Robustness summary (conf >= 0.25, IoU >= 0.5, {len(REF)} reference persons) ---")
for _, r in vru_df.iterrows():
    print(f"  {r['condition']:11s}: recall {r['recall']:.2f}, TP {r['TP']}, FN {r['FN']}, FP {r['FP']}, mean TP conf {r['mean conf (TP)']}")
worst = vru_df.loc[vru_df['recall'].idxmin()]
print(f"  Worst condition: {worst['condition']} (recall {worst['recall']:.2f}).")
occ = vru_df.set_index('condition').loc['occluded']
if occ['FN'] > 0:
    print("  Occlusion: the partly hidden person is either missed or detected with a truncated box that fails IoU >= 0.5"
          " - a 'detection' that is useless for distance and path estimation.")

In [ ]:
gains = [1.0, 0.12, 0.06, 0.03]
night_keys = ['normal', 'night_0.12', 'night_0.06', 'night_0.03']
rec = [vru_df.set_index('condition').loc[k, 'recall'] for k in night_keys]
conf_mean = [np.mean([c for c, _ in results[k]]) if results[k] else 0.0 for k in night_keys]
fig, ax1 = plt.subplots(figsize=(8, 4.5))
ax1.plot(gains, rec, 'o-', color=OKABE_ITO[4], label='Recall (conf >= 0.25, IoU >= 0.5)')
ax1.plot(gains, [max([c for c, _ in results[k]], default=0) for k in night_keys], 's--', color=OKABE_ITO[5],
         label='Highest person confidence')
ax1.set_xscale('log'); ax1.invert_xaxis()
ax1.set_xlabel('Relative exposure (1 = original image, log scale)'); ax1.set_ylabel('Recall / confidence')
ax1.set_ylim(-0.05, 1.05); ax1.legend(); ax1.set_title('Person detection vs simulated darkness (synthetic transform)')
plt.tight_layout(); plt.show()
for g, r_, k in zip(gains, rec, night_keys):
    print(f"  exposure {g:<5}: recall {r_:.2f}, person detections (any conf >= 0.05): {len(results[k])}")
drop = next((g for g, r_ in zip(gains, rec) if r_ < 1.0), None)
print(f"  Recall first drops below 1.0 at exposure {drop}; it reaches {rec[-1]:.2f} at exposure {gains[-1]}.")

**Observation:** the detector degrades gracefully at first (lower confidence, looser boxes) and then fails abruptly. A synthetic darkening is not a real night image (no headlights, no motion blur, no noise model of a real sensor), so treat this as a qualitative demonstration of a **triggering condition**, not as a performance figure. Note also that hand-drawn cartoon "pedestrians" are out of distribution for a COCO-trained model and are typically not detected at all — a lesson in itself about test data.

---
## 3. Failure modes and mitigations

> **Illustrative data:** the weights and severities below are synthetic, chosen for teaching; they are not measurements.

In [ ]:
failure_modes = pd.DataFrame({
    'Failure mode': ['Occlusion', 'Low light / night', 'Motion blur', 'Small / distant objects',
                     'Unusual poses', 'Weather (rain/fog/glare)', 'Crowded scenes', 'Non-standard VRUs'],
    'Illustrative weight': [30, 25, 15, 20, 10, 20, 15, 8],
    'Illustrative severity (1-10)': [9, 10, 7, 8, 6, 9, 7, 10],
    'Mitigation': ['Multi-view fusion, temporal tracking', 'Thermal camera, LiDAR fusion', 'Shorter exposure, higher frame rate',
                   'Higher-resolution cameras, LiDAR', 'More diverse training data', 'Radar fusion, sensor cleaning',
                   'Tracking, NMS tuning / NMS-free detectors', 'Targeted data collection and augmentation'],
})
display(failure_modes)

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(failure_modes['Illustrative weight'], failure_modes['Illustrative severity (1-10)'], s=200, color=OKABE_ITO[4], alpha=0.8)
for _, r in failure_modes.iterrows():
    ax.annotate(r['Failure mode'], (r['Illustrative weight'], r['Illustrative severity (1-10)']),
                xytext=(6, 6), textcoords='offset points', fontsize=9)
w_thr, s_thr = 15, 7
ax.axhline(s_thr, color=OKABE_ITO[5], ls='--'); ax.axvline(w_thr, color=OKABE_ITO[5], ls='--')
ax.set_xlabel('Illustrative frequency weight (relative)'); ax.set_ylabel('Illustrative severity (1-10)')
ax.set_title('VRU detection failure modes (illustrative, not measured data)')
plt.tight_layout(); plt.show()

print("NOTE: values are illustrative teaching numbers, not measurements.")
high = failure_modes[(failure_modes['Illustrative weight'] > w_thr) & (failure_modes['Illustrative severity (1-10)'] > s_thr)]
print(f"Failure modes above both thresholds (weight > {w_thr}, severity > {s_thr}):")
for _, r in high.iterrows():
    print(f"  - {r['Failure mode']}: mitigation = {r['Mitigation']}")

---
## 4. ISO 26262: determining ASILs correctly

ISO 26262-3:2018 classifies each **hazardous event** (hazard × operational situation) by

- **Severity** S0–S3 (S1 light and moderate injuries, S2 severe and life-threatening injuries — survival probable, S3 life-threatening injuries — survival uncertain — and fatal injuries),
- **Exposure** E0–E4 (probability of the operational situation),
- **Controllability** C0–C3 (C1 simply controllable, C2 normally controllable, C3 difficult to control or uncontrollable),

and reads the ASIL from Table 4. **QM** (quality management) is *not* an ASIL: it means normal QM processes suffice. Note that hazards are rated at **vehicle level** — "no braking for a crossing pedestrian" — not at the level of a fault such as "sensor failure".

In [ ]:
# ISO 26262-3:2018 Table 4. ASIL_TABLE[S][E] -> (C1, C2, C3)
# The same function is available as avsafety.iso26262.determine_asil (see repo avsafety/).
ASIL_TABLE = {
    "S1": {"E1": ("QM", "QM", "QM"), "E2": ("QM", "QM", "QM"), "E3": ("QM", "QM", "A"), "E4": ("QM", "A", "B")},
    "S2": {"E1": ("QM", "QM", "QM"), "E2": ("QM", "QM", "A"), "E3": ("QM", "A", "B"), "E4": ("A", "B", "C")},
    "S3": {"E1": ("QM", "QM", "A"), "E2": ("QM", "A", "B"), "E3": ("A", "B", "C"), "E4": ("B", "C", "D")},
}
def determine_asil(s, e, c):
    if s == "S0" or e == "E0" or c == "C0":
        return "QM"
    return ASIL_TABLE[s][e][int(c[1]) - 1]

assert determine_asil("S3", "E4", "C3") == "D" and determine_asil("S2", "E4", "C2") == "B"
assert determine_asil("S1", "E3", "C1") == "QM" and determine_asil("S3", "E2", "C3") == "B"

S_TXT = {'S1': 'S1 (light/moderate)', 'S2': 'S2 (severe, survival probable)', 'S3': 'S3 (life-threatening/fatal)'}
C_TXT = {'C1': 'C1 (simply controllable)', 'C2': 'C2 (normally controllable)', 'C3': 'C3 (difficult/uncontrollable)'}
# Illustrative HARA rows for an urban AEB/ADS braking function. The S/E/C ratings are examples
# chosen for teaching; a real HARA justifies each rating for the specific item and situation.
events = [
    ('No braking for crossing pedestrian (urban, 50 km/h)',        'S3', 'E4', 'C3'),
    ('No braking for crossing pedestrian (car park, < 10 km/h)',   'S2', 'E4', 'C2'),
    ('Late braking for pedestrian (residual speed ~20 km/h)',      'S2', 'E3', 'C2'),
    ('Unintended full braking on motorway (rear-end risk)',        'S2', 'E4', 'C2'),
    ('Unintended light braking in slow traffic',                   'S1', 'E3', 'C1'),
]
asil_df = pd.DataFrame([{'Hazardous event': h, 'Severity': S_TXT[s], 'Exposure': e,
                         'Controllability': C_TXT[c], 'ASIL': determine_asil(s, e, c)} for h, s, e, c in events])
display(asil_df)
print("--- HARA examples (text summary; ratings illustrative) ---")
for (h, s, e, c), a in zip(events, asil_df['ASIL']):
    print(f"  {h}: {s}/{e}/{c} -> {a}")
print("Note: unintended braking (a false positive) can itself reach ASIL B here - phantom braking is a safety issue,"
      " not only a comfort issue (cf. NHTSA PE22-002 on unexpected braking).")

### Example (illustrative) safety requirements for a pedestrian-detection function

ISO 26262 does **not** prescribe sensor sets, recall values or test mileages. The following are typical design measures and *example* requirements a team might derive — not quotations from the standard:

1. **Redundancy / diversity:** independent sensing paths (e.g. camera + radar or LiDAR) where the safety concept relies on ASIL decomposition (ISO 26262-9) or on SOTIF risk reduction.
2. **Diagnostics:** sensor health monitoring (blockage, calibration drift, frame drops).
3. **Degraded modes:** defined behaviour when a sensor is lost (e.g. reduce speed, minimal-risk manoeuvre).
4. **Verification and validation:** requirements-based testing, HIL/SIL, scenario-based testing, field operation. RAND (Kalra & Paddock 2016) showed that *hundreds of millions to billions of miles* may be needed to demonstrate fatality-rate superiority statistically — which is why scenario-based arguments are needed (see [06_AI_Safety_ISO_PAS_8800/07 — Statistical safety validation](../06_AI_Safety_ISO_PAS_8800/07_statistical_safety_validation.ipynb)).
5. **Performance targets** (illustrative): per-class recall within a distance band at a defined operating threshold; maximum time-to-first-detection; bounded latency.

---
## 5. Safety-oriented metrics

> **Illustrative data:** the target values below are examples for teaching, not values from a standard.

In [ ]:
safety_metrics = pd.DataFrame({
    'Metric': ['Recall (per class, per distance band)', 'Precision', 'End-to-end latency',
               'False positives per image (FPPI) / per hour', 'Track-level recall / time to first detection',
               'Detection range', 'Robustness (per triggering condition)'],
    'Example target (illustrative)': ['>= 99 % within 40 m at the operating threshold', '>= 95 %', '< 100 ms',
                                      '< 1 unwarranted brake intervention per 10^4 h', 'detected within 0.5 s of becoming visible',
                                      '> stopping distance + margin', 'no unacceptable drop in any condition within the ODD'],
    'Why it matters': ['Missed VRU -> potential collision', 'False alarms -> unnecessary braking, rear-end risk, distrust',
                       'Late detection leaves no time to brake', 'FPR is undefined for detection (no true negatives)',
                       'Single missed frames are tolerable if the track survives', 'Braking distance grows with v^2',
                       'Rare conditions dominate residual risk'],
})
display(safety_metrics)
print("NOTE: the targets are illustrative examples, not normative values.")
for _, r in safety_metrics.iterrows():
    print(f"  {r['Metric']}: {r['Example target (illustrative)']} -- {r['Why it matters']}")

### Choosing the operating threshold for a recall target

We simulate detector scores for 2,000 pedestrian instances and 2,000 background candidates and compute the precision–recall trade-off.

> **Illustrative data:** simulated scores, chosen for teaching; not measurements.

In [ ]:
pos = rng.beta(6, 2, 2000)            # scores of real pedestrians
neg = rng.beta(2, 6, 2000)            # scores of background candidates
thr = np.linspace(0.0, 1.0, 501)
recall = np.array([(pos >= t).mean() for t in thr])
fp = np.array([(neg >= t).sum() for t in thr]); tp = recall * len(pos)
precision = np.where(tp + fp > 0, tp / np.maximum(tp + fp, 1e-12), 1.0)
f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-12)
i_f1 = int(np.argmax(f1))
target = 0.999
ok = np.where(recall >= target)[0]
i_safe = ok[-1] if len(ok) else None

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
axes[0].plot(recall, precision, color=OKABE_ITO[4], lw=2, label='Simulated detector')
axes[0].scatter(recall[i_f1], precision[i_f1], marker='*', s=300, color=OKABE_ITO[0], zorder=5,
                label=f'Max F1 = {f1[i_f1]:.3f} (thr {thr[i_f1]:.2f})')
if i_safe is not None:
    axes[0].scatter(recall[i_safe], precision[i_safe], marker='D', s=120, color=OKABE_ITO[5], zorder=5,
                    label=f'Recall >= {target} (thr {thr[i_safe]:.3f})')
axes[0].set_xlabel('Recall'); axes[0].set_ylabel('Precision'); axes[0].set_xlim(0.5, 1.001); axes[0].set_ylim(0.4, 1.01)
axes[0].set_title('Simulated precision-recall curve (illustrative)'); axes[0].legend(loc='lower left')
for y, lab, col in [(recall, 'Recall', OKABE_ITO[4]), (precision, 'Precision', OKABE_ITO[2]), (f1, 'F1', OKABE_ITO[0])]:
    axes[1].plot(thr, y, label=lab, color=col, lw=2)
axes[1].axhline(target, color=OKABE_ITO[5], ls='--', label=f'Recall target {target}')
axes[1].set_xlabel('Confidence threshold'); axes[1].set_ylabel('Score'); axes[1].set_ylim(0, 1.02)
axes[1].set_title('Metrics vs confidence threshold (illustrative)'); axes[1].legend()
plt.tight_layout(); plt.show()

print(f"Max-F1 operating point: threshold {thr[i_f1]:.2f}, recall {recall[i_f1]:.3f}, precision {precision[i_f1]:.3f}")
if i_safe is not None:
    print(f"Highest threshold reaching recall >= {target}: {thr[i_safe]:.3f} -> precision {precision[i_safe]:.3f} "
          f"({int(fp[i_safe])} false positives per {len(pos)} pedestrians)")
else:
    print(f"No threshold reaches recall >= {target}.")
print("The F1 optimum is NOT the safety optimum: the recall target forces a much lower threshold, and the resulting"
      " false positives must then be handled (fusion, tracking) because phantom braking is also hazardous.")

---
## 6. Exercise: from recall to risk

**Scenario:** after testing on 1 million frames the pedestrian detector produced, over 50,000 pedestrian *instances* (a pedestrian visible in a frame): 49,500 TP, 1,000 FP and 500 FN.

1. Compute precision and recall.
2. Does it meet an (illustrative) target of recall ≥ 99.9 %?
3. What does a miss mean physically?
4. What would you improve?

In [ ]:
tp_, fp_, fn_ = 49_500, 1_000, 500
n_instances = tp_ + fn_
precision_ = tp_ / (tp_ + fp_); recall_ = tp_ / (tp_ + fn_); f1_ = 2 * precision_ * recall_ / (precision_ + recall_)
print(f"Precision {precision_:.4f}, recall {recall_:.4f}, F1 {f1_:.4f}, miss rate {fn_/n_instances:.4f}")
target = 0.999
print(f"Recall target {target}: {'met' if recall_ >= target else 'NOT met'} "
      f"({fn_} missed pedestrian instances (frame-level) per {n_instances:,}; target allows {int(n_instances*(1-target))})")

fps, v = 30, 50 / 3.6
print(f"\nDistance travelled while a pedestrian goes unseen at 50 km/h ({fps} fps camera):")
for k in (1, 3, 10, 30):
    print(f"  {k:2d} consecutive missed frame(s) = {k * v / fps:5.2f} m")
print(f"Expected missed pedestrian-frames per second at {fps} fps if one pedestrian is always visible: {fps * fn_ / n_instances:.2f}")

### Frame-level vs track-level recall

Whether 1 % of missed frames matters depends on **how misses are distributed in time**. If misses are independent, a tracker that needs, say, 3 hits out of 5 frames almost never loses a pedestrian; if misses come in long bursts (a pedestrian behind a parked van, or in glare), the track is lost for exactly as long as the burst lasts. Tracking is the subject of [07 Multi-object tracking](07_multi_object_tracking.ipynb).

> **Illustrative data:** simulated miss sequences, chosen for teaching.

In [ ]:
def simulate_misses(n_frames, p_miss, burst_len, rng):
    """Binary miss sequence with average miss rate p_miss; misses come in bursts of mean length burst_len."""
    if burst_len <= 1:
        return rng.random(n_frames) < p_miss
    p_exit = 1 / burst_len                          # two-state Markov chain (seen <-> missed)
    p_enter = p_miss * p_exit / (1 - p_miss)
    m = np.zeros(n_frames, bool); state = False
    for i in range(n_frames):
        state = (rng.random() >= p_exit) if state else (rng.random() < p_enter)
        m[i] = state
    return m

def longest_run(m):
    best = cur = 0
    for x in m:
        cur = cur + 1 if x else 0; best = max(best, cur)
    return best

n_frames, p = 30 * 60 * 60, 0.01             # 1 hour at 30 fps, 1 % frame misses
rows = []
for burst in [1, 5, 15, 45]:
    m = simulate_misses(n_frames, p, burst, np.random.default_rng(burst))
    lr = longest_run(m)
    rows.append({'mean burst length (frames)': burst, 'observed miss rate': round(m.mean(), 4),
                 'longest blind run (frames)': lr, 'longest blind distance at 50 km/h (m)': round(lr * v / fps, 1)})
burst_df = pd.DataFrame(rows); display(burst_df)
for r in rows:
    print(f"  burst {r['mean burst length (frames)']:2d}: miss rate {r['observed miss rate']:.4f}, longest blind run "
          f"{r['longest blind run (frames)']} frames = {r['longest blind distance at 50 km/h (m)']} m at 50 km/h")
rates = [r['observed miss rate'] for r in rows]
print(f"Similar frame-level miss rates ({min(rates):.2%}-{max(rates):.2%}), very different safety: "
      f"the longest blind interval grows from {rows[0]['longest blind distance at 50 km/h (m)']} m to "
      f"{rows[-1]['longest blind distance at 50 km/h (m)']} m when misses are correlated.")

### Required detection range

A pedestrian must be detected early enough to stop. With reaction/latency time $t_r$ and deceleration $a$, the stopping distance is $d_{stop} = v\,t_r + v^2 / (2a)$.

> **Illustrative data:** $t_r$ = 0.5 s (perception, decision and brake build-up) and $a$ = 6 m/s² are assumptions for teaching.

In [ ]:
t_r, a = 0.5, 6.0
speeds = np.array([30, 50, 70, 100, 130])
d_stop = speeds / 3.6 * t_r + (speeds / 3.6) ** 2 / (2 * a)
for s, d in zip(speeds, d_stop):
    print(f"  {s:3d} km/h: stopping distance {d:6.1f} m (+ tracking confirmation time and margin)")
fig, ax = plt.subplots(figsize=(8, 4.5))
vv = np.linspace(10, 130, 100)
ax.plot(vv, vv / 3.6 * t_r + (vv / 3.6) ** 2 / (2 * a), color=OKABE_ITO[4], lw=2)
ax.set_xlabel('Speed (km/h)'); ax.set_ylabel('Required detection distance (m)')
ax.set_title(f'Stopping distance (t_r = {t_r} s, a = {a} m/s², illustrative)')
plt.tight_layout(); plt.show()
print("Combine with notebook 03: at the distances needed above ~100 km/h a pedestrian receives only a handful of LiDAR"
      " points and few camera pixels - long-range VRU detection is a key performance limitation.")

---
## Safety perspective

| Observation in this notebook | Standard | Typical measure |
|---|---|---|
| Missed pedestrian in darkness, fog, occlusion (detector works as designed) | ISO 21448 — performance insufficiency, triggering condition | Sensor diversity, scenario-based validation, ODD restriction |
| Too few training examples of wheelchair users / children; cartoon figures not recognised | ISO/PAS 8800 — data and model insufficiency | Data specification and coverage analysis, targeted collection |
| Camera fails, frame drops, corrupted memory | ISO 26262 — E/E malfunction | Diagnostics, redundancy, ASIL decomposition |
| Unintended braking on a false positive | ISO 26262 (if caused by a fault) or ISO 21448 (if caused by a performance limitation) | Plausibility checks, fusion, tracking confirmation |

**Industry practice:** most L4 operators (e.g. Waymo) combine camera, LiDAR and radar, but no regulation or standard mandates a specific sensor set, and camera-centric approaches exist — Tesla began a camera-based robotaxi service in Austin in June 2025 ([TechCrunch](https://techcrunch.com/2025/06/22/tesla-launches-robotaxi-rides-in-austin-with-big-promises-and-unanswered-questions)).

Continue with: [06 Sensor fusion](06_sensor_fusion.ipynb) · [03_Functional_Safety_ISO26262/02 — HARA](../03_Functional_Safety_ISO26262/02_hara.ipynb) · [04_SOTIF_ISO21448/02 — Triggering conditions](../04_SOTIF_ISO21448/02_triggering_conditions_and_scenarios.ipynb) · [02_Failures_and_Robustness/01 — Incident case studies](../02_Failures_and_Robustness/01_incident_case_studies.ipynb).

## Key takeaways

- In the US, ~78 % of pedestrian fatalities in 2022 occurred in the dark (NHTSA DOT HS 813 590).
- On a real street image, YOLO11n degrades gradually and then abruptly under synthetic darkness, fog and occlusion; occluded persons get truncated boxes that fail the IoU criterion.
- ASILs are assigned to hazardous events via S/E/C (ISO 26262-3 Table 4) — not to 'pedestrian detection'; QM is not an ASIL; phantom braking can reach an ASIL too.
- Detector misses due to performance limitations are SOTIF (ISO 21448) issues; data/model-specific causes are addressed by ISO/PAS 8800.
- The max-F1 threshold is not the safety optimum; a recall target forces a lower threshold and more false positives that must be handled.
- Frame-level recall hides temporal structure: correlated misses produce long blind intervals, measured in metres travelled.
- Required detection range follows from stopping distance; long-range VRU detection is a key limitation.

## Self-assessment

1. Why can a safety goal involving pedestrian collisions reach ASIL D, and why are missed detections caused by model limitations (not faults) mainly a SOTIF / ISO/PAS 8800 issue?
2. Determine the ASIL for S2 / E4 / C2 and for S1 / E3 / C1 using ISO 26262-3 Table 4.
3. A system achieves 99.0 % recall on pedestrian instances. How many instances would be missed per million pedestrian instances, and why is this number alone not enough to judge safety?
4. Why is the false positive *rate* undefined for object detection, and what is used instead?
5. In the burst simulation, all four cases have ~1 % missed frames. Why are they not equally safe?
6. Describe three failure modes that affect VRU detection and propose a mitigation for each.

<details>
<summary>Show answers</summary>

1. A collision with a pedestrian at urban speed can be fatal (S3), the situation is frequent (E4) and hard for a driver to control (C3) → ASIL D. A detector that misses a pedestrian in fog without any fault is not a malfunction in the ISO 26262 sense; it is a performance insufficiency (ISO 21448), and if it stems from training data or model limitations ISO/PAS 8800 applies.
2. S2/E4/C2 → ASIL B; S1/E3/C1 → QM.
3. 10,000 missed instances per million. Safety depends on how misses are distributed in time (track loss), distance (late detection), conditions and the remaining mitigations (tracking, fusion, AEB), not only on the average.
4. There are no well-defined true negatives (every background region could be a negative). Use false positives per image (FPPI), per hour or per kilometre.
5. Correlated misses form long bursts: the pedestrian is invisible for many consecutive frames (up to tens of metres of travel), and trackers lose the track; independent misses are bridged by tracking.
6. Occlusion → multi-view fusion and tracking; darkness → thermal camera or LiDAR/radar fusion; small/distant objects → higher-resolution sensors and range-aware requirements (others: crowds → NMS tuning, unusual VRUs → targeted data).

</details>

## References

1. NHTSA. *Traffic Safety Facts 2022 Data: Pedestrians.* DOT HS 813 590. https://crashstats.nhtsa.dot.gov/Api/Public/ViewPublication/813590
2. NTSB (2019). HAR-19/03, Tempe, Arizona, March 18, 2018. https://www.ntsb.gov/investigations/AccidentReports/Reports/HAR1903.pdf
3. NHTSA (2024). Consent order — Cruise crash reporting. https://www.nhtsa.gov/press-releases/consent-order-cruise-crash-reporting
4. NHTSA ODI (2024). PE24-031 opening resume (FSD, reduced roadway visibility). https://static.nhtsa.gov/odi/inv/2024/INOA-PE24031-23232.pdf
5. NHTSA ODI (2022). PE22-002 opening resume (unexpected braking). https://static.nhtsa.gov/odi/inv/2022/INOA-PE22002-4385.PDF
6. ISO 26262-3:2018. *Road vehicles — Functional safety — Part 3: Concept phase.* https://www.iso.org/standard/68385.html
7. ISO 21448:2022. *Road vehicles — Safety of the intended functionality.* https://www.iso.org/standard/77490.html
8. ISO/PAS 8800:2024. *Road vehicles — Safety and artificial intelligence.*
9. Kalra, N., Paddock, S. M. (2016). *Driving to Safety.* RAND RR-1478. https://www.rand.org/pubs/research_reports/RR1478.html
10. Patel, M., Jung, R., Khatun, M. (2025). A Systematic Literature Review on Safety of the Intended Functionality for Automated Driving Systems. SAE Technical Paper 2025-01-5030. https://doi.org/10.4271/2025-01-5030

---
**Navigation:** ← [04 Object detection](04_object_detection.ipynb) · [Module index](README.md) · [06 Sensor fusion](06_sensor_fusion.ipynb) →